# Use Workflows to Evaluate the Accuracy of Agent Responses

An **offline LLM eval** pipeline on top of Elasticsearch Workflows, using the **LLM-as-a-Judge** pattern:

- **Stage 1 (Answer):** a RAG agent answers a question from a knowledge base. We use **Claude Haiku 4.5** here (small, cheap, fast).
- **Stage 2 (Judge):** **Claude Sonnet 4.6** scores the answer against the ground truth on correctness, faithfulness, and context relevance.

The question we want to answer with this eval: *is Haiku good enough on this task to ship in production instead of Sonnet?* The judge tells us.

## Setup

In [ ]:
%pip install -q elasticsearch==9.4 python-dotenv==1.0.1 requests==2.32.3 datasets==4 matplotlib==3.10.0

In [ ]:
from dotenv import load_dotenv
import os
import json
import requests

load_dotenv()

ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
KIBANA_URL = os.getenv("KIBANA_URL")

## Connect to Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY, request_timeout=30
)
es_client.info()

## AI Connectors

Two roles, both native Anthropic connectors:

| Role | Connector |
| :--- | :--- |
| Answering model (stage 1) | Anthropic Claude Haiku 4.5 |
| Judge model (stage 2) | Anthropic Claude Sonnet 4.6 |

In [ ]:
ANSWER_MODEL = "Anthropic Claude Haiku 4.5"
JUDGE_MODEL = "Anthropic Claude Sonnet 4.6"

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

## Load and index the dataset

We use [hotpot_qa](https://huggingface.co/datasets/hotpotqa/hotpot_qa) in the `distractor` configuration. Each question ships with 10 context paragraphs: 2 that support the answer, 8 distractors. Questions are multi-hop, meaning the answer requires combining information from two paragraphs. This is the kind of task where a weak model falls behind a strong one.

We sample a small subset of questions, flatten their context paragraphs into the knowledge base, and store the QA pairs in the judgement list.

In [ ]:
from datasets import load_dataset

SAMPLE_SIZE = 50

qa_full = load_dataset(
    "hotpot_qa", "distractor", split="validation", trust_remote_code=True
)
qa = qa_full.select(range(SAMPLE_SIZE))


def build_corpus(ds):
    seen = {}

    for item in ds:
        for title, sentences in zip(
            item["context"]["title"], item["context"]["sentences"]
        ):
            passage = " ".join(sentences).strip()
            if not passage:
                continue
            key = (title, passage[:120])

            if key not in seen:
                seen[key] = {"title": title, "passage": passage}

    return list(seen.values())


corpus = build_corpus(qa)

print(f"QA pairs: {len(qa)}")
print(f"Unique context passages: {len(corpus)}")
print("QA example:", {"question": qa[0]["question"], "answer": qa[0]["answer"]})
print("Corpus example:", corpus[0])

### Knowledge base: context passages

`hotpot-knowledge-base` holds the context passages the answering model retrieves from. Each document has a `title` and a `passage`:

```json
{"title": "Ed Wood (film)", "passage": "Ed Wood is a 1994 American biographical period comedy-drama film..."}
```

The `passage` field is copied to `semantic_content` (mapped as `semantic_text`) so Stage 1 can retrieve relevant passages by question without managing embeddings manually.

In [ ]:
INDEX_NAME = "hotpot-knowledge-base"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "title": {"type": "keyword"},
            "passage": {
                "type": "text",
                "copy_to": "semantic_content",
            },
            "semantic_content": {
                "type": "semantic_text",
                "inference_id": ".jina-embeddings-v5-text-small",
            },
        }
    },
)
print(f"Created index: {INDEX_NAME}")

In [ ]:
from elasticsearch import helpers


def build_corpus_actions(passages, index_name):
    for i, item in enumerate(passages):
        yield {
            "_index": index_name,
            "_id": str(i),
            "_source": {
                "title": item["title"],
                "passage": item["passage"],
            },
        }


success, _ = helpers.bulk(
    es_client, build_corpus_actions(corpus, INDEX_NAME), refresh=True
)
print(f"Indexed {success} passages into '{INDEX_NAME}'")

### Judgement list: questions and expected answers

`hotpot-judgement-list` holds the evaluation cases. Each document has a `question` and an `answer` — the expected final answer (the ground truth):

```json
{"question": "Were Scott Derrickson and Ed Wood of the same nationality?", "answer": "yes"}
```

The `answer` field is a short string (not a passage). The judge in Stage 2 compares the answering model's output against it. Stage 1 never sees the ground truth — it retrieves from the knowledge base and generates an answer just as it would in production. The ground truth is only a reference point for scoring.

In [ ]:
JUDGE_INDEX = "hotpot-judgement-list"

if es_client.indices.exists(index=JUDGE_INDEX):
    es_client.indices.delete(index=JUDGE_INDEX)

es_client.indices.create(
    index=JUDGE_INDEX,
    mappings={
        "properties": {
            "question": {"type": "text"},
            "answer": {"type": "text"},
            "qid": {"type": "keyword"},
        }
    },
)


def build_qa_actions(ds, index_name):
    for item in ds:
        yield {
            "_index": index_name,
            "_id": str(item["id"]),
            "_source": {
                "question": item["question"],
                "answer": item["answer"],
                "qid": str(item["id"]),
            },
        }


success, _ = helpers.bulk(es_client, build_qa_actions(qa, JUDGE_INDEX), refresh=True)
print(f"Indexed {success} QA pairs into '{JUDGE_INDEX}'")

## Workflow definition

We define the workflow YAML as a Python string and upload it via the Workflows API (Elastic 9.4+). The judge step uses **structured output** (`schema`): the model is forced to return a typed JSON object, so the `save` step can write each score to its own field. No markdown fences, no regex, no parsing on read.

In [ ]:
WORKFLOW_YAML = """
name: agent_accuracy_eval
description: >
  Batch evaluation. Loads the judgement list, iterates each case with
  a foreach, runs stage 1 (RAG answer with Haiku) and stage 2 (LLM
  judge with Sonnet), and indexes every score into the eval results
  index.
enabled: true

consts:
  kbIndex: hotpot-knowledge-base
  judgeIndex: hotpot-judgement-list
  resultsIndex: eval-results

triggers:
  - type: manual

steps:
  # Load the judgement list (all cases we want to evaluate).
  - name: load_cases
    type: elasticsearch.search
    with:
      index: "{{ consts.judgeIndex }}"
      query:
        match_all: {}
      size: 50

  # One iteration = one evaluation.
  - name: eval_loop
    type: foreach
    foreach: "{{ steps.load_cases.output.hits.hits }}"
    steps:
      - name: retrieve
        type: elasticsearch.search
        with:
          index: "{{ consts.kbIndex }}"
          query:
            semantic:
              field: semantic_content
              query: "{{ foreach.item._source.question }}"
          size: 4

      - name: agent_answer
        type: ai.prompt
        connector-id: Anthropic-Claude-Haiku-4-5
        with:
          prompt: >
            You are a Wikipedia QA assistant. Answer the question
            using ONLY the passages provided. Keep the answer short
            (one line). If the passages do not contain the answer,
            reply "unknown".

            Question: {{ foreach.item._source.question }}

            Passages:
            1. {{ steps.retrieve.output.hits.hits[0]._source.passage }}
            2. {{ steps.retrieve.output.hits.hits[1]._source.passage }}
            3. {{ steps.retrieve.output.hits.hits[2]._source.passage }}
            4. {{ steps.retrieve.output.hits.hits[3]._source.passage }}

      - name: judge
        type: ai.prompt
        connector-id: Anthropic-Claude-Sonnet-4-6
        with:
          prompt: >
            You are a STRICT evaluator. Score the candidate answer
            against the ground truth on three axes. Each score MUST
            be exactly one of these three values: 0.0, 0.5, or 1.0.
            Do not return any other number.

            correctness:
              1.0 = candidate contains the ground truth answer exactly
                    or an unambiguous synonym, and nothing factually wrong.
              0.5 = partially correct (one side of a multi-hop right,
                    or mostly right with minor noise).
              0.0 = wrong, missing, or contradicts the ground truth.

            faithfulness:
              1.0 = every factual claim is supported by the passages.
              0.5 = mostly supported, one minor unsupported claim.
              0.0 = contains at least one unsupported claim.

            context_relevance:
              1.0 = the passages contain enough to answer the question.
              0.5 = partial coverage (one hop covered, the other missing).
              0.0 = passages do not cover the answer.

            Be harsh. If in doubt between two scores, pick the lower one.

            Question: {{ foreach.item._source.question }}
            Ground truth: {{ foreach.item._source.answer }}
            Candidate: {{ steps.agent_answer.output.content }}
            Passages:
            1. {{ steps.retrieve.output.hits.hits[0]._source.passage }}
            2. {{ steps.retrieve.output.hits.hits[1]._source.passage }}
            3. {{ steps.retrieve.output.hits.hits[2]._source.passage }}
            4. {{ steps.retrieve.output.hits.hits[3]._source.passage }}
          schema:
            type: object
            properties:
              correctness:
                type: number
                minimum: 0
                maximum: 1
              faithfulness:
                type: number
                minimum: 0
                maximum: 1
              context_relevance:
                type: number
                minimum: 0
                maximum: 1
            required:
              - correctness
              - faithfulness
              - context_relevance

      # Persist each scored case so the notebook can query it later.
      - name: save
        type: elasticsearch.index
        with:
          index: "{{ consts.resultsIndex }}"
          document:
            qid: "{{ foreach.item._source.qid }}"
            question: "{{ foreach.item._source.question }}"
            ground_truth: "{{ foreach.item._source.answer }}"
            candidate: "{{ steps.agent_answer.output.content }}"
            correctness: "{{ steps.judge.output.content.correctness }}"
            faithfulness: "{{ steps.judge.output.content.faithfulness }}"
            context_relevance: "{{ steps.judge.output.content.context_relevance }}"
"""

### What the judge measures

The judge returns three scores, each between 0.0 and 1.0. Together they tell you *where* the agent fails.

| Metric | Question it answers | What a low score means |
| :--- | :--- | :--- |
| **correctness** | Does the candidate answer match the ground truth? | The agent answered the wrong thing. |
| **faithfulness** | Is the answer grounded in the retrieved passages? | The agent hallucinated (made something up not in the context). |
| **context_relevance** | Are the retrieved passages relevant to the question? | Retrieval brought the wrong documents. Bad query or bad index. |

A high `correctness` with low `context_relevance` is a red flag: the model answered correctly despite poor retrieval, usually by falling back on its own training data. That works for Wikipedia, but breaks on private data the model has never seen.

## Upload and run the workflow via the Workflows API

Elastic 9.4 ships a REST API for workflows. We use three endpoints:

- `POST /api/workflows` to create the workflow from YAML
- `POST /api/workflows/{id}/run` to start an execution
- `GET /api/workflows/executions/{id}` to poll until it finishes

### Create the results index

The workflow writes one scored case per document. Scores are typed numerics, so we map them as `float` and use `keyword`/`text` for the rest.

In [ ]:
RESULTS_INDEX = "eval-results"

if es_client.indices.exists(index=RESULTS_INDEX):
    es_client.indices.delete(index=RESULTS_INDEX)

es_client.indices.create(
    index=RESULTS_INDEX,
    mappings={
        "properties": {
            "qid": {"type": "keyword"},
            "question": {"type": "text"},
            "ground_truth": {"type": "text"},
            "candidate": {"type": "text"},
            "correctness": {"type": "float"},
            "faithfulness": {"type": "float"},
            "context_relevance": {"type": "float"},
        }
    },
)
print(f"Created index: {RESULTS_INDEX}")

### Create the workflow

In [ ]:
workflow_headers = {
    **headers,
    "x-elastic-internal-origin": "Kibana",
}

r = requests.post(
    f"{KIBANA_URL}/api/workflows",
    headers=workflow_headers,
    json={"workflows": [{"yaml": WORKFLOW_YAML}]},
    params={"overwrite": "true"},
)
r.raise_for_status()
created = r.json()
WORKFLOW_ID = created["created"][0]["id"]
print(f"Created workflow: {WORKFLOW_ID}")

### Run it and wait for completion

In [ ]:
import time

TERMINAL_STATUSES = {"completed", "failed", "cancelled", "error"}


def wait_for_execution(
    execution_id: str, poll_seconds: int = 5, timeout_minutes: int = 30
) -> str:
    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        try:
            r = requests.get(
                f"{KIBANA_URL}/api/workflows/executions/{execution_id}",
                headers=workflow_headers,
                timeout=15,
            )
            r.raise_for_status()
            status = r.json().get("status", "").lower()
            print(f"  status: {status}")
            if status in TERMINAL_STATUSES:
                return status
        except (requests.ConnectionError, requests.Timeout) as e:
            print(f"  transient: {type(e).__name__}, retrying...")
        time.sleep(poll_seconds)
    raise TimeoutError(
        f"Execution {execution_id} did not finish in {timeout_minutes} min"
    )


r = requests.post(
    f"{KIBANA_URL}/api/workflows/workflow/{WORKFLOW_ID}/run",
    headers=workflow_headers,
    json={"inputs": {}},
)
r.raise_for_status()
EXECUTION_ID = r.json()["workflowExecutionId"]
print(f"Started execution: {EXECUTION_ID}")

final_status = wait_for_execution(EXECUTION_ID)
if final_status != "completed":
    raise RuntimeError(f"Workflow ended with status {final_status}")
print("Workflow finished")

## Read results

Each scored case is already a document in `eval-results` with typed score fields. No parsing needed: we read straight from the index.

In [ ]:
import matplotlib.pyplot as plt

es_client.indices.refresh(index=RESULTS_INDEX)
hits = es_client.search(index=RESULTS_INDEX, query={"match_all": {}}, size=1000)[
    "hits"
]["hits"]
print(f"Loaded {len(hits)} result documents")

metrics = ["correctness", "faithfulness", "context_relevance"]
parsed = [h["_source"] for h in hits]


def to_float(v) -> float:
    try:
        return float(v) if v not in (None, "") else 0.0
    except (TypeError, ValueError):
        return 0.0


means = {
    m: round(sum(to_float(p.get(m)) for p in parsed) / max(len(parsed), 1), 3)
    for m in metrics
}
print("Mean scores:", json.dumps(means, indent=2))

### Aggregated scores

In [ ]:
import numpy as np

x = np.arange(len(metrics))

fig, ax = plt.subplots(figsize=(7, 4))
vals = [means[m] for m in metrics]
bars = ax.bar(x, vals, color=["#1565C0", "#2E7D32", "#EF6C00"])

for b, v in zip(bars, vals):
    ax.text(
        b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9
    )

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score (0-1)")
ax.set_title(f"Mean judge scores ({ANSWER_MODEL} as answering model)")

plt.tight_layout()
plt.show()

### Next step: compare with a stronger model

To compare Haiku against a stronger answering model, change `connectorId` in the `agent_answer` step from `Anthropic Claude Haiku 4.5` to `Anthropic Claude Sonnet 4.6`, re-run the workflow, and overlay the bar charts. 

## Cleanup

In [ ]:
requests.delete(f"{KIBANA_URL}/api/workflows/{WORKFLOW_ID}", headers=workflow_headers)
print(f"Deleted workflow: {WORKFLOW_ID}")

es_client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
es_client.indices.delete(index=JUDGE_INDEX, ignore_unavailable=True)
es_client.indices.delete(index=RESULTS_INDEX, ignore_unavailable=True)
print("Cleaned up indices")